This visualizes a matrix of escapes and homings in each condition, sorted by the tuning curve for that condition calculated across escapes and homings

In [2]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

#JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
# 
experiments_objects = [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, # JAL6_flip7_1apr, # this session is sus
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_flip4_10may, JAL8_14may]

#
trials = [[1,3],[1,5,7],[1],[2],
    [2,3],[1,3],
    [1,3,4],[1,2,3],[1,2],[1,3],
    [4,5],[1,3],[3,4],[4,5],[2,5],
    [1,3],[1,2],[1,3],[5,7,8],[3,5],
]

In [1]:
%load_ext autoreload
from behave_analysis.process.process import Process
from behave_analysis.utils.creating_directories import make_directory
from JR_test_scripts.escape.escape_utils import load, check_not_list, compute_dist_shelt, compute_escape_trajectory, compress_vars, discretize_x_axis, firing_by_bin

import os
import polars as pl
import dill as pickle
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
%matplotlib inline

In [105]:
%autoreload
%matplotlib inline
compression_var = ['y_pos', 'distance_shelter', 'escape','speed']
for i, exp in enumerate(experiments_objects[2:]):
    session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip = load(exp)
    ons, offs = load_homing(session)
    for comp in compression_var:
        nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp
        esc_var, escape_matrix, start, h_start, cond = extract_homing_time(session, ons, offs, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, comp)
        tuning_cond = tuning_by_condition(esc_var, escape_matrix, cond)
        plot_tuning_matrix(tuning_cond, cond, session, comp, escape_matrix, esc_var, start, h_start)

In [86]:
def load_homing(session):
    homie_path = os.path.join(session.base_path, session.processed_path, "homings", "homings_obj.pkl")
    with open(homie_path, "rb") as dill_file:
        homings = pickle.load(dill_file)
    return homings.onset_frames, homings.offset_frames

In [85]:
def extract_homing_time(session, ons, offs, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, compression_var):
    """Tuning for each neuron is measured by compression_var"""
    # extract the time around escapes
    start = [0]
    esc_start = [0]
    h_start = [0]

    esc_ons = check_not_list(session.audio.onset_frames)
    st = [x*40 for x in check_not_list(session.audio.stimulus_durations)]
    esc_offs = (np.add(esc_ons, st)).astype(int)

    ons = np.sort(np.append(check_not_list(ons), esc_ons))
    offs = np.sort(np.append(check_not_list(offs), esc_offs))

    escape_matrix = np.zeros((np.shape(frame_by_cluster_matrix)[1],np.sum(offs - ons)))
    esc_var = np.zeros(np.sum(offs - ons))
    in_shelter = np.zeros(np.sum(offs - ons))
    cond = np.zeros(np.sum(offs - ons))
    for tr, (on,of) in enumerate(zip(ons, offs)):
        # find actual length of time until mouse is in shelter (or 5s if he never makes it)
        y_loc = y_pos[on:of]
        x_loc = x_pos[on:of]
        in_shelt_y = y_loc > session.shelter_location[0][1]
        in_shelt_x = np.logical_and(x_loc > session.shelter_location[0][0],x_loc < session.shelter_location[1][0])
        in_shelt = np.logical_and(in_shelt_x, in_shelt_y)

        # extract variables
        neur = frame_by_cluster_matrix[on:of,:] # time x neurons
        this_speed = behave[on:of]
        this_y = y_pos[on:of]
        this_x = x_pos[on:of]

        # condition vector
        c = np.zeros((len(this_y)))
        if bar[on] == True: c += 1
        if barflip[on] == True: c += 1

        bin_size = 10
        if compression_var == 'distance_shelter':
            var = compute_dist_shelt(this_x, this_y, c, session)
        elif compression_var == 'y_pos':
            var = this_y
        elif compression_var == 'escape':
            dd = compute_escape_trajectory(this_x, this_y)
            var = (dd/np.amax(dd))
            bin_size = .01
        elif compression_var == 'speed':
            var = this_speed
            bin_size = 1

        disc_var = discretize_x_axis(var, bin_size)

        # concatenate trials
        escape_matrix[:,start[-1]:start[-1]+len(this_y)] = neur.T
        esc_var[start[-1]:start[-1]+len(this_y)] = disc_var
        in_shelter[start[-1]:start[-1]+len(this_y)] = in_shelt
        cond[start[-1]:start[-1]+len(this_y)] = c
        
        if on in esc_ons:
            esc_start.append(h_start[-1])
        start.append(start[-1]+len(this_y))
        if len(np.where(in_shelt)[0]) == 0:        
            h_start.append(h_start[-1]+len(this_y))
        else:
            h_start.append(h_start[-1]+len(disc_var[in_shelt == 0]))
        
    if np.amin(ons) < np.amin(esc_ons):
        esc_start = esc_start[1:]
    cond = cond[in_shelter == 0]
    esc_var = esc_var[in_shelter == 0]
    escape_matrix = escape_matrix[:,in_shelter == 0]
    escape_matrix = zscore(escape_matrix, axis = 1) # neurons x time
    return esc_var, escape_matrix, esc_start, h_start, cond

In [84]:
def creat_tuning_curve(esc_var, escape_matrix):
    tuning_matrix = np.empty((np.shape(escape_matrix)[0],len(np.unique(esc_var))))
    for i, n in enumerate(escape_matrix):
        tuning_matrix[i,:] = firing_by_bin(esc_var.astype(int), n, int(np.amax(esc_var)+1))
    return tuning_matrix

In [83]:
def tuning_by_condition(esc_var, escape_matrix, cond):
    tuning_by_cond = []
    for i in np.unique(cond):
        tuning_matrix = creat_tuning_curve(esc_var[cond == i], escape_matrix[:,cond == i])
        tuning_by_cond.append(tuning_matrix)
    return tuning_by_cond

In [104]:
def plot_tuning_matrix(tuning_matrix, cond, session, compression_var, escape_matrix, esc_var, start, h_start):

    condy = ['shelter only', 'barrier', 'flipped barrier']
    fig = plt.figure(figsize=(40,22), dpi=200)
    grid = plt.GridSpec(26, 20, figure=fig, wspace = 0.05, hspace = 0.3)
    
    esss_var = np.zeros_like(esc_var)
    for it, st in enumerate(h_start):
        if st in start:
            esss_var[st:h_start[it+1]] = esc_var[st:h_start[it+1]]
    esss_var[esss_var == 0] = np.nan

    for i, lim in enumerate([[1,8],[10,17],[19,26]]):
        idx = np.argmax(tuning_matrix[i], axis = 1)
        isort = np.argsort(idx)
        # tuning curve, by condition
        ax = plt.subplot(grid[lim[0]:lim[1],:2])
        ax.imshow(tuning_matrix[i][isort,:], cmap="gray_r", vmin = 0, vmax = 1.2, aspect="auto", interpolation = "none")
        ax.set_ylabel('neurons')
        ax.set_ylim([0,len(isort)])
        ax.set_xlabel(compression_var)
        ax.set_title('tuning curve')
        ax.set_ylabel('sorted by tuning in ' + condy[i])

        # behavioral var
        ax = plt.subplot(grid[lim[0]-1, 3:])
        e = esc_var[cond == i]
        ax.plot(esc_var[cond == i], 'b')
        ax.plot(esss_var[cond == i], 'r')
        ax.set_xlim([0,len(esc_var[cond == i])])
        ax.axis("off")
        ax.set_ylabel(compression_var)

        # neural data
        ax = plt.subplot(grid[lim[0]:lim[1], 3:])
        e = escape_matrix[:,cond == i]
        ax.imshow(e[isort,:], cmap="gray_r", vmin = 0, vmax = 1.2, aspect="auto", interpolation = "none")
        for st in h_start[:-1]:
            if cond[st] == i:
                st -= np.where(cond == i)[0][0]
                if st in start:
                    ax.plot([st,st],[len(isort),0],'--r', linewidth = .7)
                else:
                    ax.plot([st,st],[len(isort),0],'--b', linewidth = .7)
        ax.set_xlabel('time')
        ax.set_yticks([])
        ax.set_ylim([0,len(isort)])
        ax.set_xlim([0,len(esc_var[cond == i])])

    # stim_resp_path = make_directory(os.path.join(session.base_path, session.processed_path, "stim_resp", "rastermap","tuning"))
    # save_path=str(stim_resp_path) + "/" + nickname + ".png"
    # fig.savefig(save_path)

    dump_path = "Z:/Jasmine_Laurence/homing/tuning"
    fig.savefig(dump_path + "/" + nickname + ".png")
    plt.close()